# PLFS Wage Dataset Harmonization (2023-2025)

This notebook builds the model-ready wage dataset from three PLFS person-level
files. It is the first notebook in the project and should be run before
`02_wage_model_end_to_end.ipynb`.

**Inputs** (place them in `data/raw/`):

- `data/raw/2023.csv` - PLFS July 2023-June 2024, first-visit person file
- `data/raw/2024.csv` - PLFS Calendar Year 2024 person file
- `data/raw/2025.csv` - PLFS Calendar Year 2025 person file

See `README.md` for download links and the expected file layout.

**Outputs:**

- `data/processed/plfs_2023_2024_2025_harmonized.csv`
- `data/processed/wage_merge_summary.csv`
- `data/processed/wage_merge_report.json`
- `figures/data/*.png` - data-audit figures for slides

The notebook keeps the original benchmark's year-specific quirks (including the
2025 education mapping and 69 retained duplicate rows) because the model
documented here was trained with them.


In [ ]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)

# Work from either the repository root or notebooks/.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = ROOT / "data" / "raw"
OUT_DIR = ROOT / "data" / "processed"
FIG_DIR = ROOT / "figures" / "data"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

PATH_2023_RAW = str(DATA_DIR / "2023.csv")
PATH_2024_RAW = str(DATA_DIR / "2024.csv")
PATH_2025_RAW = str(DATA_DIR / "2025.csv")

OUTPUT_CSV = str(OUT_DIR / "plfs_2023_2024_2025_harmonized.csv")

CPI_FACTOR_2023 = 1.1372
CPI_FACTOR_2024 = 1.0794
CPI_FACTOR_2025 = 1.0505

REGULAR_WAGE_STATUS = "31"

for p in (PATH_2023_RAW, PATH_2024_RAW, PATH_2025_RAW):
    if not Path(p).exists():
        raise FileNotFoundError(p)

print("Configuration loaded.")
print("Repository root:", ROOT)
print("Output:", OUTPUT_CSV)


In [ ]:

NUMERIC_FEATURES = [
    "age",
    "potential_experience",
    "potential_experience_sq",
    "years_formal_education",
    "tothrs_wrk",
    "days_worked",
    "avg_daily_hours",
]

CATEGORICAL_FEATURES = [
    "sex",
    "marst",
    "rel",
    "gedu_lvl",
    "tedu_lvl",
    "curr_att",
    "secondary",
    "voc",
    "voc_fld",
    "voc_typ",
    "st",
    "dc",
    "state_district",
    "sec",
    "qtr",
    "nss_reg",
    "sro",
    "ind_pas",
    "ocu_pas",
    "loc_pas",
    "etyp_pas",
    "wrkr_pas",
    "job_pas",
    "leave_pas",
    "ssec_pas",
    "has_sas",
    "acws",
    "aind_cws",
    "ocu_cws",
    "dur_pas",
]

OUTPUT_COLUMNS = [
    "survey_year",
    "ern_reg_nominal",
    "cpi_factor",
    "ern_reg_current",
] + NUMERIC_FEATURES + CATEGORICAL_FEATURES

print("Final columns:", len(OUTPUT_COLUMNS))


In [ ]:


def clean_code(series):
    s = (
        series.astype("string")
        .str.strip()
        .replace({
            "": pd.NA,
            "nan": pd.NA,
            "None": pd.NA,
            "<NA>": pd.NA,
        })
    )

    n = pd.to_numeric(
        s,
        errors="coerce"
    )

    mask = s.notna() & n.notna()

    out = s.copy()

    out.loc[mask] = (
        n.loc[mask]
        .astype("Int64")
        .astype("string")
    )

    return out


def regular_worker_filter(df):
    df["pas"] = clean_code(df["pas"])

    df["ern_reg"] = pd.to_numeric(
        df["ern_reg"],
        errors="coerce"
    )

    df = df[
        df["pas"] == REGULAR_WAGE_STATUS
    ].copy()

    df = df[
        df["ern_reg"] > 0
    ].copy()

    return df


def finalize_common(df, year, cpi_factor):
    df = df.copy()

    df["survey_year"] = year

    df["ern_reg_nominal"] = pd.to_numeric(
        df["ern_reg"],
        errors="coerce"
    )

    df["cpi_factor"] = float(
        cpi_factor
    )

    df["ern_reg_current"] = (
        df["ern_reg_nominal"]
        *
        df["cpi_factor"]
    )

    for col in CATEGORICAL_FEATURES:
        if col in df.columns:
            df[col] = clean_code(df[col])

    for col in NUMERIC_FEATURES + [
        "ern_reg_nominal",
        "ern_reg_current",
        "survey_year",
        "cpi_factor",
    ]:
        if col in df.columns:
            df[col] = pd.to_numeric(
                df[col],
                errors="coerce"
            )

    missing = [
        c for c in OUTPUT_COLUMNS
        if c not in df.columns
    ]

    if missing:
        raise ValueError(
            f"{year}: missing common columns:\n"
            + "\n".join(missing)
        )

    return df[OUTPUT_COLUMNS].copy()


## 2023: map the raw `perv1` file into the common schema


In [ ]:

raw23 = pd.read_csv(
    PATH_2023_RAW,
    low_memory=False
)

MAP_2023 = {
    "b4q6_perv1": "age",
    "b4q5_perv1": "sex",
    "b4q7_perv1": "marst",
    "b4q4_perv1": "rel",
    "b4q8_perv1": "gedu_lvl",
    "b4q9_perv1": "tedu_lvl",
    "b4q10_perv1": "years_formal_education",
    "b4q11_perv1": "curr_att",
    "b4q12_perv1": "voc",
    "b4pt1q4_perv1": "voc_fld",
    "b4pt1q6_perv1": "voc_typ",

    "state_perv1": "st",
    "distcode_perv1": "dc",
    "b1q3_perv1": "sec",
    "qtr_perv1": "qtr",
    "nss_region_perv1": "nss_reg",
    "b1q12_perv1": "sro",

    "b5pt1q5_perv1": "ind_pas",
    "b5pt1q6_perv1": "ocu_pas",
    "b5pt1q7_perv1": "has_sas",
    "b5pt1q8_perv1": "loc_pas",
    "b5pt1q9_perv1": "etyp_pas",
    "b5pt1q10_perv1": "wrkr_pas",
    "b5pt1q11_perv1": "job_pas",
    "b5pt1q12_perv1": "leave_pas",
    "b5pt1q13_perv1": "ssec_pas",
    "b5pt3q6_perv1": "dur_pas",

    "b6q5_perv1": "acws",
    "b6q6_perv1": "aind_cws",
    "b6q7_perv1": "ocu_cws",
}

missing = [
    c for c in MAP_2023
    if c not in raw23.columns
]

if missing:
    raise ValueError(
        "2023 source columns missing:\n"
        + "\n".join(missing)
    )

df23 = raw23.rename(
    columns=MAP_2023
).copy()

print("2023 mapping complete.")


In [ ]:


df23["age"] = pd.to_numeric(
    df23["age"],
    errors="coerce"
)

df23["years_formal_education"] = pd.to_numeric(
    df23["years_formal_education"],
    errors="coerce"
)

df23["potential_experience"] = np.maximum(
    0,
    df23["age"]
    - df23["years_formal_education"]
    - 6
)

df23["potential_experience_sq"] = (
    df23["potential_experience"] ** 2
)

for col in CATEGORICAL_FEATURES:
    if col in df23.columns:
        df23[col] = clean_code(df23[col])

df23["secondary"] = (
    df23["gedu_lvl"]
    .isin({"8", "08", "10", "11", "12", "13"})
    .astype("Int64")
    .astype("string")
)

hour_cols23 = [
    "b6q7_3pt1_perv1",
    "b6q7_3pt2_perv1",
    "b6q7_3pt3_perv1",
    "b6q7_3pt4_perv1",
    "b6q7_3pt5_perv1",
    "b6q7_3pt6_perv1",
    "b6q7_3pt7_perv1",
]

missing_hours = [
    c for c in hour_cols23
    if c not in raw23.columns
]

if missing_hours:
    raise ValueError(
        "2023 daily-hour columns missing:\n"
        + "\n".join(missing_hours)
    )

hours23 = (
    raw23[hour_cols23]
    .apply(pd.to_numeric, errors="coerce")
    .fillna(0)
)

df23["tothrs_wrk"] = hours23.sum(axis=1)
df23["days_worked"] = (hours23 > 0).sum(axis=1)
df23["avg_daily_hours"] = hours23.mean(axis=1)

df23["state_district"] = (
    df23["st"].astype("string")
    + "_"
    + df23["dc"].astype("string")
)
df23.loc[df23["st"].isna() | df23["dc"].isna(), "state_district"] = pd.NA

df23["pas"] = clean_code(
    raw23["b5pt1q3_perv1"]
)

df23["ern_reg"] = pd.to_numeric(
    raw23["b6q9_perv1"],
    errors="coerce"
)

before = len(df23)

df23 = regular_worker_filter(df23)

df23 = finalize_common(
    df23,
    2023,
    CPI_FACTOR_2023
)
# Keep duplicates.  The original Dev benchmark
# (Dev/Benchmark Models/combind 2023-2025) contains 69 duplicate rows
# (10 + 12 + 47).  The older (2024+2025)+2023 route retained them; dropping
# them changes the row population and is the source of the 213,172-row drift.

print(
    f"2023: {before:,} raw rows -> "
    f"{len(df23):,} final rows"
)


## 2024: map and prepare the raw 2024 person-level file


In [ ]:


raw24 = pd.read_csv(
    PATH_2024_RAW,
    low_memory=False
)

MAP_2024 = {
    "State_UT_Code": "st",
    "District_Code": "dc",
    "NSS_Region": "nss_reg",
    "FOD_Sub_Region": "sro",
    "Sector": "sec",
    "Quarter": "qtr",

    "Relationship_To_Head": "rel",
    "Sex": "sex",
    "Age": "age",
    "Marital_Status": "marst",
    "General_Education_Level": "gedu_lvl",
    "Technical_Education_Level": "tedu_lvl",
    "Years_Formal_Education": "years_formal_education",
    "Current_Attendance_Status": "curr_att",

    "Vocational_Training": "voc",
    "Field_Of_Training": "voc_fld",
    "Type_Of_Training": "voc_typ",

    "Principal_Status_Code": "pas",
    "Principal_Industry_Code": "ind_pas",
    "Principal_Occupation_Code": "ocu_pas",
    "Subsidiary_Work_Engagement": "has_sas",
    "Principal_Workplace_Location": "loc_pas",
    "Principal_Enterprise_Type": "etyp_pas",
    "Principal_Workers_Count": "wrkr_pas",
    "Principal_Job_Contract_Type": "job_pas",
    "Principal_Paid_Leave": "leave_pas",
    "Principal_Social_Security": "ssec_pas",
    "Principal_Duration": "dur_pas",

    "CWS_Status_Code": "acws",
    "CWS_Industry_Code": "aind_cws",
    "CWS_Occupation_Code": "ocu_cws",
    "CWS_Earnings_Salaried": "ern_reg",
}

missing = [
    c for c in MAP_2024
    if c not in raw24.columns
]

if missing:
    raise ValueError(
        "2024 source columns missing:\n"
        + "\n".join(missing)
    )

df24 = raw24.rename(
    columns=MAP_2024
).copy()

print("2024 mapping complete.")


In [ ]:


df24["age"] = pd.to_numeric(
    df24["age"],
    errors="coerce"
)

df24["years_formal_education"] = pd.to_numeric(
    df24["years_formal_education"],
    errors="coerce"
)

df24["potential_experience"] = np.maximum(
    0,
    df24["age"]
    - df24["years_formal_education"]
    - 6
)

df24["potential_experience_sq"] = (
    df24["potential_experience"] ** 2
)

for col in CATEGORICAL_FEATURES:
    if col in df24.columns:
        df24[col] = clean_code(df24[col])

# Original 2024 benchmark behavior: the old route tested the zero-padded
# set {"08", "10", "11", "12", "13"} against normalized codes, so only
# 10-13 matched and code 8 stayed 0.  Reproduce that exactly.
df24["secondary"] = (
    df24["gedu_lvl"]
    .isin({"10", "11", "12", "13"})
    .astype("Int64")
    .astype("string")
)



hour_cols24 = [
    "Day1_Total_Hours", "Day2_Total_Hours", "Day3_Total_Hours",
    "Day4_Total_Hours", "Day5_Total_Hours", "Day6_Total_Hours",
    "Day7_Total_Hours",
]
# NOTE: do NOT use substring search for 2024 hours. The raw file order
# starts with Day7_Act1_Hours, Day7_Act2_Hours, Day7_Total_Hours, ... so
# [:7] silently picks act-level hours and collapsed 2024 tothrs
# (mean ~30 vs ~48 in 2023/2025). These must be the seven daily TOTALS.
missing_hours_24 = [c for c in hour_cols24 if c not in raw24.columns]
if missing_hours_24:
    raise ValueError(
        "2024 daily-total hour columns missing:\n"
        + "\n".join(missing_hours_24)
    )
hours24 = (
    raw24[hour_cols24]
    .apply(pd.to_numeric, errors="coerce")
    .fillna(0)
)
df24["tothrs_wrk"] = hours24.sum(axis=1)
df24["days_worked"] = (hours24 > 0).sum(axis=1)
df24["avg_daily_hours"] = hours24.mean(axis=1)

df24["state_district"] = (
    df24["st"].astype("string")
    + "_"
    + df24["dc"].astype("string")
)
df24.loc[df24["st"].isna() | df24["dc"].isna(), "state_district"] = pd.NA

before = len(df24)

df24 = regular_worker_filter(df24)

df24 = finalize_common(
    df24,
    2024,
    CPI_FACTOR_2024
)
print(
    f"2024: {before:,} raw rows -> "
    f"{len(df24):,} final rows"
)


## 2025: prepare the raw `cperv12025` file


In [ ]:


df25 = pd.read_csv(
    PATH_2025_RAW,
    low_memory=False
)

required25 = [
    "age",
    "gedu_lvl",
    "pas",
    "ind_pas",
    "ocu_pas",
    "st",
    "dc",
    "ern_reg",
]

missing = [
    c for c in required25
    if c not in df25.columns
]

if missing:
    raise ValueError(
        "2025 required columns missing:\n"
        + "\n".join(missing)
    )

print("2025 loaded.")


In [ ]:


for col in CATEGORICAL_FEATURES:
    if col in df25.columns:
        df25[col] = clean_code(df25[col])

df25["age"] = pd.to_numeric(
    df25["age"],
    errors="coerce"
)

# 2025 education: reproduce the original benchmark exactly.
#
# The original 2025 harmonization used a zero-padded education map
# ("01".."08").  `gedu_lvl` is normalized to unpadded codes ("1".."8")
# before the map is applied, so codes 1-8 intentionally stay NaN and only
# 10-13 resolve (10 -> 15, 11/12/13 -> 17).  This is the behavior baked
# into the trained wage model and the Dev benchmark; do not "fix" it
# without retraining and re-validating the model.
EDU_YEARS_25 = {
    "01": 0, "02": 0, "03": 3, "04": 5,
    "05": 8, "06": 10, "07": 12, "08": 14,
    "10": 15, "11": 17, "12": 17, "13": 17,
}

df25["years_formal_education"] = (
    df25["gedu_lvl"]
    .map(EDU_YEARS_25)
)

df25["potential_experience"] = np.maximum(
    0,
    df25["age"]
    - df25["years_formal_education"]
    - 6
)

df25["potential_experience_sq"] = (
    df25["potential_experience"] ** 2
)

# Original 2025 benchmark behavior: `secondary` is the raw 2025 column
# (codes 1/2/3 with NaN for most rows), not a derived gedu_lvl indicator.
# It was already normalized by the categorical loop above; leave it as-is.

# 2025 hours: compute all three from hr1..hr7 exactly like 2023/2024
# (sum / count>0 / mean). Verified tothrs_wrk == sum(hr1..hr7) (corr 1.0)
# on raw 2025, so the provided column is only a cross-check.
hour_cols_25 = [f"hr{i}" for i in range(1, 8)]
missing_hour_cols = [
    c
    for c in hour_cols_25
    if c not in df25.columns
]

if missing_hour_cols:
    raise ValueError(
        "2025 raw file is missing daily-hour columns:\n"
        + "\n".join(missing_hour_cols)
    )

hours25 = (
    df25[hour_cols_25]
    .apply(
        pd.to_numeric,
        errors="coerce"
    )
    .fillna(0)
)
if "tothrs_wrk" in df25.columns:
    provided25 = pd.to_numeric(df25["tothrs_wrk"], errors="coerce").fillna(0)
    mismatch25 = (provided25 - hours25.sum(axis=1)).abs()
    print(f"2025 tothrs cross-check: max |provided - sum(hr1..7)| = {mismatch25.max():.6f}")
    if mismatch25.max() > 1e-6:
        raise ValueError("2025 tothrs_wrk disagrees with sum(hr1..hr7).")
df25["tothrs_wrk"] = hours25.sum(axis=1)
df25["days_worked"] = (hours25 > 0).sum(axis=1)
df25["avg_daily_hours"] = hours25.mean(axis=1)

print(
    df25[
        [
            "tothrs_wrk",
            "days_worked",
            "avg_daily_hours"
        ]
    ].describe()
)
df25["state_district"] = (
    df25["st"].astype("string")
    + "_"
    + df25["dc"].astype("string")
)
df25.loc[df25["st"].isna() | df25["dc"].isna(), "state_district"] = pd.NA

before = len(df25)

df25 = regular_worker_filter(df25)

df25 = finalize_common(
    df25,
    2025,
    CPI_FACTOR_2025
)
print(
    f"2025: {before:,} raw rows -> "
    f"{len(df25):,} final rows"
)


## Final direct three-way merge

All three years are now in the same 41-column schema.

We concatenate rows directly:

`2023 + 2024 + 2025`

No cross-year join or cross-year deduplication is performed.


In [ ]:


final_df = pd.concat(
    [
        df23,
        df24,
        df25,
    ],
    ignore_index=True
)

final_df = final_df[
    OUTPUT_COLUMNS
].copy()

print("=" * 80)
print("FINAL DIRECT MERGE")
print("=" * 80)

print(
    "Shape:",
    final_df.shape
)

print()
print(
    "Rows by survey_year:"
)

print(
    final_df[
        "survey_year"
    ]
    .value_counts()
    .sort_index()
)

print()
print(
    "CPI factors:"
)

print(
    final_df.groupby(
        "survey_year"
    )["cpi_factor"]
    .first()
    .sort_index()
)


In [ ]:


assert list(
    final_df.columns
) == OUTPUT_COLUMNS

assert (
    final_df["survey_year"]
    .isin([2023, 2024, 2025])
    .all()
)

assert (
    final_df["ern_reg_nominal"] > 0
).all()

assert (
    final_df["ern_reg_current"] > 0
).all()

# CPI config guard: project base requires 2023 > 2024 > 2025 > 0.
assert CPI_FACTOR_2023 > CPI_FACTOR_2024 > CPI_FACTOR_2025 > 0
for _yr, _fac in [(2023, CPI_FACTOR_2023), (2024, CPI_FACTOR_2024), (2025, CPI_FACTOR_2025)]:
    assert (final_df.loc[final_df["survey_year"] == _yr, "cpi_factor"] == _fac).all()

# Harmonization audit: hours and schooling must be comparable across years.
# 2024 hours bug (act-level [:7]) showed median tothrs ~32 vs ~48;
# 2025 edu bug (overstated mapping) showed median yrs ~15 vs ~12.
audit = final_df.groupby("survey_year").agg(
    med_tothrs=("tothrs_wrk", "median"),
    med_yrs=("years_formal_education", "median"),
    med_ern_nom=("ern_reg_nominal", "median"),
)
print(audit)
assert final_df.groupby("survey_year")["tothrs_wrk"].median().between(40, 56).all(), audit

# 2025 education quirk: the original benchmark maps only codes 10-13,
# leaving codes 1-8 as NaN (see the 2025 education cell).  Guard the exact
# distribution so an accidental "fix" cannot silently change the training
# data.
yrs25 = final_df.loc[final_df["survey_year"] == 2025, "years_formal_education"]
assert int(yrs25.isna().sum()) == 51735, int(yrs25.isna().sum())
assert set(yrs25.dropna().unique()) == {15.0, 17.0}, sorted(yrs25.dropna().unique())

expected_med_yrs = {2023: 12.0, 2024: 12.0, 2025: 17.0}
med_yrs = final_df.groupby("survey_year")["years_formal_education"].median().to_dict()
assert {int(k): float(v) for k, v in med_yrs.items()} == expected_med_yrs, med_yrs

# secondary follows the original year-specific behavior:
#   2023 derived from {8,10,11,12,13}
#   2024 derived from {10,11,12,13} (the old zero-padded "08" missed code 8)
#   2025 raw column (1/2/3/NaN)
m23 = final_df["survey_year"] == 2023
m24 = final_df["survey_year"] == 2024
m25 = final_df["survey_year"] == 2025
exp23 = final_df.loc[m23, "gedu_lvl"].isin(["8", "08", "10", "11", "12", "13"]).astype(int).astype(str)
assert (final_df.loc[m23, "secondary"].astype(str) == exp23).all()
exp24 = final_df.loc[m24, "gedu_lvl"].isin(["10", "11", "12", "13"]).astype(int).astype(str)
assert (final_df.loc[m24, "secondary"].astype(str) == exp24).all()
sec25 = pd.to_numeric(final_df.loc[m25, "secondary"], errors="coerce")
assert int(sec25.isna().sum()) == 113607, int(sec25.isna().sum())
assert set(sec25.dropna().unique()) == {1.0, 2.0, 3.0}, sorted(sec25.dropna().unique())

# The original benchmark retains duplicate rows.  Check the exact row
# population so a silent dedup cannot reintroduce the 213,172-row drift.
expected_rows = {2023: 45569, 2024: 46034, 2025: 121638}
actual_rows = {
    int(k): int(v)
    for k, v in final_df["survey_year"].value_counts().sort_index().items()
}
assert actual_rows == expected_rows, actual_rows
assert len(final_df) == 213241, len(final_df)
assert int(final_df.duplicated().sum()) == 69, int(final_df.duplicated().sum())

assert (
    len(final_df) ==
    len(df23)
    + len(df24)
    + len(df25)
)

print("=" * 80)
print("ALL FINAL CHECKS PASSED")
print("=" * 80)


In [ ]:
# Reproduce the original benchmark's CSV representation.  The 2023 route
# canonicalized code columns to integers; the 2024/2025 route kept the raw
# float representation for several columns.  pandas infers dtypes from the
# CSV text and the wage model's OrdinalEncoder stringifies categoricals, so
# this formatting is part of the model-input contract.
FLOAT_CODE_COLS = [
    "curr_att", "secondary", "voc", "voc_fld", "voc_typ", "ind_pas",
    "ocu_pas", "loc_pas", "etyp_pas", "wrkr_pas", "job_pas",
    "leave_pas", "ssec_pas", "aind_cws", "ocu_cws", "dur_pas",
]
INT_CODE_COLS = [
    "sex", "marst", "rel", "gedu_lvl", "tedu_lvl", "st", "dc", "sec",
    "nss_reg", "sro", "has_sas", "acws",
]
FLOAT_NUM_COLS = [
    "ern_reg_nominal", "cpi_factor", "ern_reg_current",
    "potential_experience", "potential_experience_sq",
    "years_formal_education", "avg_daily_hours",
]
INT_NUM_COLS = ["age", "tothrs_wrk", "days_worked"]

def format_for_csv(df):
    parts = []
    for year in (2023, 2024, 2025):
        part = df[df["survey_year"] == year].copy()
        for col in FLOAT_CODE_COLS:
            vals = pd.to_numeric(part[col], errors="coerce")
            part[col] = vals.astype("Int64") if year == 2023 else vals.astype("float64")
        for col in INT_CODE_COLS:
            part[col] = pd.to_numeric(part[col], errors="coerce").astype("int64")
        for col in FLOAT_NUM_COLS:
            part[col] = pd.to_numeric(part[col], errors="coerce").astype("float64")
        for col in INT_NUM_COLS:
            part[col] = pd.to_numeric(part[col], errors="coerce").astype("int64")
        parts.append(part.astype(object))
    return pd.concat(parts, ignore_index=True)

out_df = format_for_csv(final_df)
out_df.to_csv(
    OUTPUT_CSV,
    index=False,
    na_rep=""
)

print("=" * 80)
print("DATASET EXPORTED")
print("=" * 80)

print("Path:", os.path.abspath(OUTPUT_CSV))
print("Rows:", f"{len(final_df):,}")
print("Columns:", len(final_df.columns))

# Compact merge report for downstream notebooks and reviewers.
summary = (
    final_df.groupby("survey_year")
    .agg(
        rows=("ern_reg_current", "size"),
        median_nominal_wage=("ern_reg_nominal", "median"),
        median_current_wage=("ern_reg_current", "median"),
        median_hours=("tothrs_wrk", "median"),
        median_education_years=("years_formal_education", "median"),
    )
    .reset_index()
)
summary.to_csv(OUT_DIR / "wage_merge_summary.csv", index=False)

report = {
    "rows": int(len(final_df)),
    "columns": list(final_df.columns),
    "rows_by_year": {
        str(int(k)): int(v)
        for k, v in final_df["survey_year"].value_counts().sort_index().items()
    },
    "cpi_factors": {
        "2023": CPI_FACTOR_2023,
        "2024": CPI_FACTOR_2024,
        "2025": CPI_FACTOR_2025,
    },
    "summary": summary.to_dict(orient="records"),
}
with open(OUT_DIR / "wage_merge_report.json", "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

print("\nMerge summary:")
print(summary.to_string(index=False))


## Presentation figures — data and merge audit

These figures summarise the harmonized wage dataset. They are saved as PNGs
under `figures/data/` for direct use in slides.


In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

PLOT_DIR = FIG_DIR
PLOT_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
})

years = summary["survey_year"].astype(int).tolist()
rows = summary["rows"].astype(int).tolist()
med_nom = summary["median_nominal_wage"].astype(float).tolist()
med_real = summary["median_current_wage"].astype(float).tolist()
med_hours = summary["median_hours"].astype(float).tolist()
med_edu = summary["median_education_years"].astype(float).tolist()

# 1) Summary table.
fig, ax = plt.subplots(figsize=(10, 2.7))
ax.axis("off")
data = [[
    str(y), f"{r:,}", f"Rs {mn:,.0f}", f"Rs {mr:,.0f}", f"{mh:.0f}", f"{me:.0f}"
] for y, r, mn, mr, mh, me in zip(years, rows, med_nom, med_real, med_hours, med_edu)]
tbl = ax.table(
    cellText=data,
    colLabels=["Year", "Rows", "Nominal wage (Rs)", "Real wage (Rs)",
               "Hours", "Edu. years"],
    loc="center", cellLoc="center",
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1, 1.55)
for (row, col), cell in tbl.get_celld().items():
    cell.set_edgecolor("#D9D9D9")
    if row == 0:
        cell.set_facecolor("#EAF2F8")
        cell.set_text_props(weight="bold")
ax.set_title("Harmonized wage dataset — merge summary", pad=12)
fig.tight_layout()
fig.savefig(PLOT_DIR / "01_data_summary_table.png", dpi=200, bbox_inches="tight")
plt.close(fig)

# 2) Rows by year.
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar([str(y) for y in years], rows, color="#4C78A8", width=0.55)
for b, v in zip(bars, rows):
    ax.text(b.get_x() + b.get_width() / 2, v, f"{v:,}", ha="center", va="bottom", fontsize=10)
ax.set_title("Wage dataset rows by survey year")
ax.set_ylabel("Rows")
ax.set_ylim(0, max(rows) * 1.15)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
fig.savefig(PLOT_DIR / "02_rows_by_year.png", dpi=200, bbox_inches="tight")
plt.close(fig)

print("Saved:", PLOT_DIR / "01_data_summary_table.png")
print("Saved:", PLOT_DIR / "02_rows_by_year.png")


In [ ]:
# 3) Median nominal vs real wage.
fig, ax = plt.subplots(figsize=(7, 4))
x = [str(y) for y in years]
ax.plot(x, med_nom, marker="o", lw=2, color="#F58518", label="Nominal")
ax.plot(x, med_real, marker="o", lw=2, color="#4C78A8", label="Real (2025 prices)")
for xi, v in zip(x, med_real):
    ax.annotate(f"Rs {v:,.0f}", (xi, v), textcoords="offset points", xytext=(0, 9),
                ha="center", fontsize=9, color="#4C78A8")
ax.text(0.02, 0.95, "Nominal median = Rs 15,000 in all years",
        transform=ax.transAxes, fontsize=9, color="#F58518")
ax.set_title("Median regular wage by survey year")
ax.set_ylabel("Rupees per month")
ax.set_ylim(min(med_real + med_nom) * 0.85, max(med_real + med_nom) * 1.16)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.25)
ax.legend(frameon=False, loc="lower right")
fig.tight_layout()
fig.savefig(PLOT_DIR / "03_median_wages.png", dpi=200, bbox_inches="tight")
plt.close(fig)

# 4) Real wage quantiles by year.
q = final_df.groupby("survey_year")["ern_reg_current"].quantile([0.10, 0.50, 0.90]).unstack()
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot([str(y) for y in q.index], q[0.10], marker="o", lw=2, label="p10", color="#B0B0B0")
ax.plot([str(y) for y in q.index], q[0.50], marker="o", lw=2, label="p50", color="#4C78A8")
ax.plot([str(y) for y in q.index], q[0.90], marker="o", lw=2, label="p90", color="#54A24B")
ax.set_title("Real regular-wage distribution by year")
ax.set_ylabel("Rupees per month (2025 prices)")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.25)
ax.legend(frameon=False, ncol=3)
fig.tight_layout()
fig.savefig(PLOT_DIR / "04_wage_quantiles.png", dpi=200, bbox_inches="tight")
plt.close(fig)

# 5) Education missingness, showing the 2025 education quirk.
miss = final_df.groupby("survey_year")["years_formal_education"].apply(
    lambda s: 100.0 * s.isna().mean()).reindex([2023, 2024, 2025])
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar([str(y) for y in miss.index], miss.values, color="#E45756", width=0.55)
for b, v in zip(bars, miss.values):
    ax.text(b.get_x() + b.get_width() / 2, v, f"{v:.1f}%", ha="center", va="bottom", fontsize=10)
ax.set_title("Missing years_formal_education by survey year")
ax.set_ylabel("% missing")
ax.set_ylim(0, max(miss.values) * 1.25)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
fig.savefig(PLOT_DIR / "05_education_missingness.png", dpi=200, bbox_inches="tight")
plt.close(fig)

print("Saved:", PLOT_DIR / "03_median_wages.png")
print("Saved:", PLOT_DIR / "04_wage_quantiles.png")
print("Saved:", PLOT_DIR / "05_education_missingness.png")
